# Deep Learning vs Classical ML Comparison

## Overview
This notebook compares EEGNet (deep learning on raw signal) with LogisticRegression (classical ML on band power features) using 5-fold cross-validation.

## Key parameters
| Parameter | Value | Meaning |
|-----------|-------|---------|
| CV folds | 5 | Number of CV folds |
| DL epochs | 30 | EEGNet epochs |
| ML features | 88 | 22 channels x 4 bands |

## 1. Install dependencies

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn torch

## 2. Load data and extract features

In [ ]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

y = np.array([0 if lab == 'left_hand' else 1 for lab in labels])
print(f"Data shape: {X.shape}")
print(f"Labels: {len(labels)}")
print(f"Classes: {np.unique(labels)}")

from scipy.signal import welch

FS = 250
BANDS = [(0.5, 4, 'delta'), (4, 8, 'theta'), (8, 13, 'alpha'), (13, 30, 'beta')]

def extract_band_power_features(X):
    n_trials, n_channels, n_samples = X.shape
    features = []
    for trial in range(n_trials):
        trial_features = []
        for ch in range(n_channels):
            freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
            for fmin, fmax, name in BANDS:
                mask = (freqs >= fmin) & (freqs <= fmax)
                power = np.trapezoid(psd[mask], freqs[mask])
                trial_features.append(power)
        features.append(trial_features)
    return np.array(features)

features = extract_band_power_features(X)
print(f"Feature matrix: {features.shape}")

## 3. Build EEGNet

In [ ]:
import torch
import torch.nn as nn

class EEGNet(nn.Module):
    def __init__(self, n_channels=22, n_samples=1001, n_classes=2, F1=8, D=2, F2=16, dropout=0.25):
        super().__init__()
        self.conv1 = nn.Conv2d(1, F1, (1, 64), padding='same')
        self.batchnorm1 = nn.BatchNorm2d(F1)
        self.depthwise = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1)
        self.batchnorm2 = nn.BatchNorm2d(F1 * D)
        self.activation = nn.ELU()
        self.pool1 = nn.AvgPool2d((1, 4))
        self.dropout1 = nn.Dropout(dropout)
        self.separable = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding='same', groups=F1 * D),
            nn.Conv2d(F1 * D, F2, (1, 1)),
        )
        self.batchnorm3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d((1, 8))
        self.dropout2 = nn.Dropout(dropout)
        dummy = torch.zeros(1, 1, n_channels, n_samples)
        out = self._features(dummy)
        self.classify = nn.Linear(out.view(-1).shape[0], n_classes)

    def _features(self, x):
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.depthwise(x)
        x = self.batchnorm2(x)
        x = self.activation(x)
        x = self.pool1(x)
        x = self.dropout1(x)
        x = self.separable(x)
        x = self.batchnorm3(x)
        x = self.activation(x)
        x = self.pool2(x)
        x = self.dropout2(x)
        return x

    def forward(self, x):
        x = self._features(x)
        x = x.view(x.size(0), -1)
        x = self.classify(x)
        return x

## 4. Cross-validation and comparison

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import torch

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

n_ch, n_s = X.shape[1], X.shape[2]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
dl_accs, ml_accs = [], []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"Fold {fold}/5")
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    torch.manual_seed(42)
    model = EEGNet(n_channels=n_ch, n_samples=n_s).to(device)
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32).unsqueeze(1)
    y_tr_t = torch.tensor(y_tr, dtype=torch.long)
    for ep in range(30):
        perm = torch.randperm(X_tr_t.shape[0])
        for s in range(0, X_tr_t.shape[0], 32):
            idx = perm[s:s+32]
            opt.zero_grad()
            loss = criterion(model(X_tr_t[idx].to(device)), y_tr_t[idx].to(device))
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        X_te_t = torch.tensor(X_te, dtype=torch.float32).unsqueeze(1)
        _, pred = torch.max(model(X_te_t.to(device)), 1)
    dl_accs.append(accuracy_score(y_te, pred.cpu().numpy()))

    scaler = StandardScaler()
    X_tr_f = scaler.fit_transform(features[train_idx])
    X_te_f = scaler.transform(features[test_idx])
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_tr_f, y_tr)
    ml_accs.append(accuracy_score(y_te, clf.predict(X_te_f)))

dl_mean, dl_std = np.mean(dl_accs), np.std(dl_accs)
ml_mean, ml_std = np.mean(ml_accs), np.std(ml_accs)
print(f"DL: {dl_mean:.4f} +/- {dl_std:.4f}")
print(f"ML: {ml_mean:.4f} +/- {ml_std:.4f}")

## 5. Interactive plot

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(go.Bar(
    x=['DL (EEGNet)', 'ML (LR + power)'],
    y=[dl_mean, ml_mean],
    error_y=dict(type='data', array=[dl_std, ml_std], visible=True),
    marker_color=['steelblue', 'coral'],
    text=[f'{dl_mean:.4f}', f'{ml_mean:.4f}'],
    textposition='outside'
))
fig.update_layout(title='Deep Learning vs Classical ML (5-fold CV)', yaxis_title='Accuracy', yaxis_range=[0, 1], width=700, height=500)
fig.show()

## What did we learn?
- EEGNet usually outperforms classical ML with handcrafted features
- Deep learning requires more compute
- Classical ML is better for small data or when interpretability matters